In [813]:
# importamos las librerías que necesitamos

# Tratamiento de datos
# -----------------------------------------------------------------------
import pandas as pd
import numpy as np
from src import soporte_datasets as sp_datasets

# Configuración
# -----------------------------------------------------------------------
pd.set_option('display.max_columns', None) # para poder visualizar todas las columnas de los DataFrames

# Gestión de los warnings
# -----------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

In [814]:
df_flight = pd.read_csv('files/Customer Flight Activity.csv') 
df_flight.head(2)

,Loyalty Number,Year,Month,Flights Booked,Flights with Companions,Total Flights,Distance,Points Accumulated,Points Redeemed,Dollar Cost Points Redeemed
0,100018,2017,1,3,0,3,1521,152.0,0,0
1,100102,2017,1,10,4,14,2030,203.0,0,0


In [815]:
df_loyalty = pd.read_csv('files/Customer Loyalty History.csv') #,index_col=0
df_loyalty.head(2)

,Loyalty Number,Country,Province,City,Postal Code,Gender,Education,Salary,Marital Status,Loyalty Card,CLV,Enrollment Type,Enrollment Year,Enrollment Month,Cancellation Year,Cancellation Month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN


Lo primero a realizar es poner todos los datos en minúsculas, y en las columnas, cambiar los espacios por guiones para un mejor tratamiento de los datos. 

In [816]:
df_flight.columns = sp_datasets.min_datos(df_flight)

In [817]:
df_flight.head(2)

,loyalty_number,year,month,flights_booked,flights_with_companions,total_flights,distance,points_accumulated,points_redeemed,dollar_cost_points_redeemed
0,100018,2017,1,3,0,3,1521,152.0,0,0
1,100102,2017,1,10,4,14,2030,203.0,0,0


In [818]:
df_loyalty.columns = sp_datasets.min_datos(df_loyalty)

In [819]:
df_loyalty.head(2)

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,NaN,NaN
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,NaN,NaN


# Gestión de valores nulos

Tal como hemos indicado en la fase inicial, existen nulos en las columnas de Salario y cancelación, vamos a tratarlos antes de la unión.

    - Para las columnas de cancelación de df_loyalty, vamos a pasar los Nulos como False, manteniendo el año y mes, ya que los valores nulos significan que el usuario sigue estando dado de alta en el sistema.

    - Para la columna Salario de df_loyalty, analizaremos patrones para ver la mejor opción.

In [820]:
df_loyalty["cancellation_year"]= sp_datasets.nulos_false_int(df_loyalty["cancellation_year"])
df_loyalty["cancellation_month"]= sp_datasets.nulos_false_int(df_loyalty["cancellation_month"])

In [821]:
df_loyalty.head(2)

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,False,False
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,False,False


In [822]:
(df_loyalty.isna().sum()/df_loyalty.shape[0]*100).sort_values(ascending=False).head(2)

#Comprobamos que solo está la columna Salario.

salary            25.321145
loyalty_number     0.000000
dtype: float64

In [823]:
df_loyalty.info()

#comprobamos que las columnas que acabamos de procesar es tipo objeto, tienen valores numericos y 
# boleanos, de momento vamos a dejarlo así por si tenemos que analizar % por mes.


<class 'pandas.DataFrame'>
RangeIndex: 16737 entries, 0 to 16736
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   loyalty_number      16737 non-null  int64  
 1   country             16737 non-null  str    
 2   province            16737 non-null  str    
 3   city                16737 non-null  str    
 4   postal_code         16737 non-null  str    
 5   gender              16737 non-null  str    
 6   education           16737 non-null  str    
 7   salary              12499 non-null  float64
 8   marital_status      16737 non-null  str    
 9   loyalty_card        16737 non-null  str    
 10  clv                 16737 non-null  float64
 11  enrollment_type     16737 non-null  str    
 12  enrollment_year     16737 non-null  int64  
 13  enrollment_month    16737 non-null  int64  
 14  cancellation_year   16737 non-null  object 
 15  cancellation_month  16737 non-null  object 
dtypes: float64(2), 

Para el salario, vamos a ver los salarios nulos con la Educación para ver si tienen patrón.

In [824]:
salario_negativo = df_loyalty[df_loyalty["salary"]<0]

In [825]:
salario_negativo["salary"].unique()
#Analizamos los salarios negativos.

array([-49830., -12497., -46683., -45962., -19325., -43234., -10605.,
       -17534., -58486., -31911., -49001., -34079.,  -9081., -46470.,
       -26322., -47310., -39503., -19332., -46303., -57297.])

In [826]:
len(salario_negativo["salary"].unique())
#Analizamos cuántos son

20

In [827]:
salario_negativo['education'].value_counts()

#los datos corresponden a la mayoría de educación Bachelor.

education
Bachelor                19
High School or Below     1
Name: count, dtype: int64

In [828]:
salario_negativo[["education","salary"]].sort_values("salary")

,education,salary
7373,Bachelor,-58486.0
16735,Bachelor,-57297.0
1082,High School or Below,-49830.0
8767,Bachelor,-49001.0
14327,Bachelor,-47310.0
2471,Bachelor,-46683.0
12596,Bachelor,-46470.0
16431,Bachelor,-46303.0
3575,Bachelor,-45962.0
4712,Bachelor,-43234.0


Ahora que tengo los salarios negativos, voy a verlos junto con la educación

In [829]:
media_salarios_limpios = df_loyalty[df_loyalty['salary'] > 0].groupby('education')['salary'].min().reset_index()
media_salarios_limpios.head()
#saco el valor mínimo de los salarios por nivel educativo.

,education,salary
0,Bachelor,15609.0
1,Doctor,48109.0
2,High School or Below,21853.0
3,Master,56414.0


Apreciamos que el valor minimo de Bachelor es 15609, y el valor negativo más bajo es 9081, seguido de 10605, interpretamos que la imputación de los datos ha habido una errata en el signo y los volvemos positivos, teniendo en cuenta que el valor de 9081 sea un posible outlier.

In [830]:
# Aplicamos el valor absoluto a la columna salary
df_loyalty['salary'] = df_loyalty['salary'].abs()

# Verificamos que ya no hay negativos
df_loyalty['salary'].min()

np.float64(9081.0)

In [831]:
df_loyalty.describe(include=np.number).T

,count,mean,std,min,25%,50%,75%,max
loyalty_number,16737.0,549735.880445,258912.132453,100018.00,326603.00,550434.00,772019.00,999986.00
salary,12499.0,79359.340907,34749.691464,9081.00,59246.50,73455.00,88517.50,407228.00
clv,16737.0,7988.896536,6860.982280,1898.01,3980.84,5780.18,8940.58,83325.38
enrollment_year,16737.0,2015.253211,1.979111,2012.00,2014.00,2015.00,2017.00,2018.00
enrollment_month,16737.0,6.669116,3.398958,1.00,4.00,7.00,10.00,12.00


Tras poner los valores positivos, vemos que la columna `Salary` tiene un rango muy amplio, desde los 9081 hasta los 407228, la mediana y media están próximas y la desviación estándar es muy alta, lo que podría interpertarse como dispersión en los datos.

In [832]:
cv = df_loyalty["salary"].std()/df_loyalty["salary"].mean()

cv

np.float64(0.43787777300716524)

Como tengo valores nulos en Salario, voy a realizar el coeficiente de variación de la columna para luego ver si sustituyendo los datos, se modifica mucho.

El cv es 0.43 (~ 43%), lo que indica que los datos son heterogéneos, hay mucha dispersión.

In [833]:
#Hago la media de salario por tipo de educación y vemos que el "College" no tiene valores.
round(df_loyalty.groupby("education")["salary"].mean(),2)

#orden de menor a mayor: high school < college < bachelor < master < doctor

education
Bachelor                 72577.25
College                       NaN
Doctor                  178608.90
High School or Below     61199.16
Master                  103757.85
Name: salary, dtype: float64

In [834]:
round(df_loyalty.groupby("education")["salary"].median(),2)

#College tiene todos los datos nulos en su nivel educativo, no podemos usarlo para crear un patron.

education
Bachelor                 71960.0
College                      NaN
Doctor                  182143.5
High School or Below     61915.0
Master                  105487.0
Name: salary, dtype: float64

Vamos a usar el CV inicial para ver cuál es la mejor opción de sustitución:

In [835]:
# Calculamos la mediana y media global de salario
mediana_global = df_loyalty["salary"].median()
media_global = df_loyalty["salary"].mean()

# Creamos dos columnas, uno imputando la media global de salario, otra la mediana global 

df_loyalty["salary_media"] = df_loyalty["salary"].fillna(media_global)
df_loyalty["salary_mediana"] = df_loyalty["salary"].fillna(mediana_global)

# Recalculamos los estadísticos
mean_imputado = df_loyalty["salary_media"].mean()
std_imputado = df_loyalty["salary_media"].std()
cv_imputado = std_imputado / mean_imputado

mean_imputado2 = df_loyalty["salary_mediana"].mean()
std_imputado2 = df_loyalty["salary_mediana"].std()
cv_imputado2 = std_imputado2 / mean_imputado2


print(f"CV Real: {cv:.2f}")
print(f"CV tras Imputar media global de Salary: {cv_imputado:.2f}")
print(f"CV tras Imputar mediana global de Salary: {cv_imputado2:.2f}")


CV Real: 0.44
CV tras Imputar media global de Salary: 0.38
CV tras Imputar mediana global de Salary: 0.39


Creemos que tanto la media como la mediana global de salary va a sesgar los perfiles de los clientes, como la educación tiene una jerarquía, vamos a sacar un punto medio entre los dos niveles que se encuentra "College"

In [836]:
#creamos un df con la media de los salarios por nivel educativo.
medianas_educacion = df_loyalty.groupby("education")["salary"].median()

# high school < college < bachelor
mediana_hschool = medianas_educacion['High School or Below']
mediana_bach = medianas_educacion['Bachelor']
sueldo_college_estimado = (mediana_hschool + mediana_bach) / 2

sueldo_college_estimado

np.float64(66937.5)

In [837]:
df_loyalty["salary_estimado"] = df_loyalty["salary"].fillna(sueldo_college_estimado)

mean_imputado3 = df_loyalty["salary_estimado"].mean()
std_imputado3= df_loyalty["salary_estimado"].std()
cv_imputado3 = std_imputado3 / mean_imputado3


print(f"CV Real: {cv:.2f}")
print(f"CV tras Imputar media global de Salary: {cv_imputado:.2f}")
print(f"CV tras Imputar mediana global de Salary: {cv_imputado2:.2f}")
print(f"CV tras Imputar mediana promedio por nivel educativo/Salary: {cv_imputado3:.2f}")

CV Real: 0.44
CV tras Imputar media global de Salary: 0.38
CV tras Imputar mediana global de Salary: 0.39
CV tras Imputar mediana promedio por nivel educativo/Salary: 0.40


el coeficiente de variación del salario promedio entre salario y educación de los niveles anterior y superior de "College" se acerca más a los datos, por lo tanto, nos quedaremos con esa opción.

In [838]:
df_loyalty.head(2)

,loyalty_number,country,province,city,postal_code,gender,education,salary,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month,salary_media,salary_mediana,salary_estimado
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,83236.0,Married,Star,3839.14,Standard,2016,2,False,False,83236.000000,83236.0,83236.0
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,NaN,Divorced,Star,3839.61,Standard,2016,3,False,False,79359.340907,73455.0,66937.5


In [839]:
df_loyalty.describe(include=np.number).T #para variables numéricas

,count,mean,std,min,25%,50%,75%,max
loyalty_number,16737.0,549735.880445,258912.132453,100018.00,326603.00,550434.000000,772019.00,999986.00
salary,12499.0,79359.340907,34749.691464,9081.00,59246.50,73455.000000,88517.50,407228.00
clv,16737.0,7988.896536,6860.982280,1898.01,3980.84,5780.180000,8940.58,83325.38
enrollment_year,16737.0,2015.253211,1.979111,2012.00,2014.00,2015.000000,2017.00,2018.00
enrollment_month,16737.0,6.669116,3.398958,1.00,4.00,7.000000,10.00,12.00
salary_media,16737.0,79359.340907,30029.311812,9081.00,63899.00,79359.340907,82940.00,407228.00
salary_mediana,16737.0,77864.294198,30138.879584,9081.00,63899.00,73455.000000,82940.00,407228.00
salary_estimado,16737.0,76213.988588,30511.295223,9081.00,63899.00,66937.500000,82940.00,407228.00


Antes de borrar columnas, vemos lo siguiente:

    - Si lo cambiamos por la media global, el 50% que antes cobraba 73k ahora pasa a 79k, la mediana se ha desplazado mucho.
    - Entre la mediana global y el promedio de nivel educativo, hay una pequeña diferencia en la desviación estándar, media y mediana, pero vamos a optar por dejar el salario estimado para mantener una coherencia de datos, donde high school < college < bachelor.

In [840]:
# Eliminamos todas las columnas provisionales de salary creadas.
df_loyalty.drop(columns=['salary', 'salary_mediana', 'salary_media'], inplace=True, errors='ignore')


In [841]:
df_loyalty.head(2)

,loyalty_number,country,province,city,postal_code,gender,education,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month,salary_estimado
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,Married,Star,3839.14,Standard,2016,2,False,False,83236.0
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,Divorced,Star,3839.61,Standard,2016,3,False,False,66937.5


In [842]:
df_loyalty.rename(columns={"salary_estimado": "salary"}, inplace=True)

In [843]:
df_loyalty.head(2)

,loyalty_number,country,province,city,postal_code,gender,education,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month,salary
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,Married,Star,3839.14,Standard,2016,2,False,False,83236.0
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,Divorced,Star,3839.61,Standard,2016,3,False,False,66937.5


In [844]:
print(f"Comprobamos nulos: {df_loyalty.isna().sum()}")

Comprobamos nulos: loyalty_number        0
country               0
province              0
city                  0
postal_code           0
gender                0
education             0
marital_status        0
loyalty_card          0
clv                   0
enrollment_type       0
enrollment_year       0
enrollment_month      0
cancellation_year     0
cancellation_month    0
salary                0
dtype: int64


Vamos ahora a tratar los valores duplicados del `df_flight`

In [845]:
df_flight[df_flight.duplicated()].value_counts().sum()

np.int64(1864)

In [846]:
df_flight[df_flight.duplicated(keep=False)].head(10)

,loyalty_number,year,month,flights_booked,flights_with_companions,total_flights,distance,points_accumulated,points_redeemed,dollar_cost_points_redeemed
41,101902,2017,1,0,0,0,0,0.0,0,0
42,101902,2017,1,0,0,0,0,0.0,0,0
226,112142,2017,1,0,0,0,0,0.0,0,0
227,112142,2017,1,0,0,0,0,0.0,0,0
477,126100,2017,1,0,0,0,0,0.0,0,0
478,126100,2017,1,0,0,0,0,0.0,0,0
566,130331,2017,1,0,0,0,0,0.0,0,0
567,130331,2017,1,0,0,0,0,0.0,0,0
659,135421,2017,1,0,0,0,0,0.0,0,0
660,135421,2017,1,0,0,0,0,0.0,0,0


Podemos ver que los duplicados coinciden en todas las columnas, por lo tanto, se procede a su eliminación.


In [847]:
df_flight = df_flight.drop_duplicates(keep="first") #no borra la primera aparición.

In [848]:
df_flight[df_flight.duplicated()].value_counts().sum()

np.int64(0)

In [849]:
df_flight.shape

(403760, 10)

In [850]:
print(403760+1864) #corresponde a las filas totales que tenía el df inicialmente, es okeey

405624


Vamos a comprobar la información de ambos df antes de la unión, por si se nos ha pasado algo por encima

In [851]:
df_flight.head()

,loyalty_number,year,month,flights_booked,flights_with_companions,total_flights,distance,points_accumulated,points_redeemed,dollar_cost_points_redeemed
0,100018,2017,1,3,0,3,1521,152.0,0,0
1,100102,2017,1,10,4,14,2030,203.0,0,0
2,100140,2017,1,6,0,6,1200,120.0,0,0
3,100214,2017,1,0,0,0,0,0.0,0,0
4,100272,2017,1,0,0,0,0,0.0,0,0


In [852]:
df_flight["year"].unique()

array([2017, 2018])

In [853]:
df_flight["month"].unique()

array([ 1,  9,  2,  3, 11,  4,  5,  7,  6,  8, 10, 12])

In [854]:
df_flight["points_redeemed"].unique()

array([  0, 341, 364, 310, 445, 312, 343, 366, 389, 292, 447, 324, 456,
       409, 436, 327, 322, 291, 323, 300, 290, 309, 325, 386, 321, 363,
       340, 670, 443, 517, 444, 328, 344, 367, 313, 333, 293, 449, 297,
       455, 372, 356, 405, 381, 466, 419, 369, 352, 482, 335, 329, 305,
       415, 396, 317, 348, 314, 334, 350, 330, 318, 298, 420, 336, 471,
       680, 441, 353, 484, 301, 374, 417, 501, 299, 398, 307, 368, 306,
       347, 439, 395, 481, 337, 382, 426, 373, 399, 424, 326, 392, 438,
       467, 480, 448, 308, 400, 376, 375, 460, 339, 385, 611, 431, 320,
       362, 404, 442, 410, 361, 319, 435, 414, 464, 477, 315, 485, 370,
       421, 349, 371, 416, 496, 510, 667, 465, 434, 346, 487, 408, 500,
       360, 378, 345, 358, 479, 380, 411, 491, 505, 446, 425, 476, 393,
       418, 332, 401, 454, 303, 594, 506, 355, 302, 403, 379, 437, 561,
       483, 597, 391, 562, 342, 407, 490, 468, 488, 457, 365, 357, 463,
       388, 413, 351, 462, 440, 493, 507, 338, 377, 428, 525, 39

In [855]:
df_flight["dollar_cost_points_redeemed"].unique()

array([ 0, 28, 30, 25, 36, 32, 24, 26, 37, 33, 35, 27, 31, 54, 42, 29, 38,
       34, 39, 55, 41, 49, 40, 48, 45, 53, 58, 44, 43, 46, 52, 47, 63, 57,
       62, 51, 50, 64, 56, 61, 65, 60, 68, 59, 66, 69, 67, 71, 70])

In [856]:
df_flight.info()

<class 'pandas.DataFrame'>
Index: 403760 entries, 0 to 405623
Data columns (total 10 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   loyalty_number               403760 non-null  int64  
 1   year                         403760 non-null  int64  
 2   month                        403760 non-null  int64  
 3   flights_booked               403760 non-null  int64  
 4   flights_with_companions      403760 non-null  int64  
 5   total_flights                403760 non-null  int64  
 6   distance                     403760 non-null  int64  
 7   points_accumulated           403760 non-null  float64
 8   points_redeemed              403760 non-null  int64  
 9   dollar_cost_points_redeemed  403760 non-null  int64  
dtypes: float64(1), int64(9)
memory usage: 33.9 MB


Una vez confirmado que df_flight tiene los datos correctos, valores unicos más de un dato, pasamos al siguiente df.

In [857]:
df_loyalty.head()

,loyalty_number,country,province,city,postal_code,gender,education,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month,salary
0,480934,Canada,Ontario,Toronto,M2Z 4K1,Female,Bachelor,Married,Star,3839.14,Standard,2016,2,False,False,83236.0
1,549612,Canada,Alberta,Edmonton,T3G 6Y6,Male,College,Divorced,Star,3839.61,Standard,2016,3,False,False,66937.5
2,429460,Canada,British Columbia,Vancouver,V6E 3D9,Male,College,Single,Star,3839.75,Standard,2014,7,2018,1,66937.5
3,608370,Canada,Ontario,Toronto,P1W 1K4,Male,College,Single,Star,3839.75,Standard,2013,2,False,False,66937.5
4,530508,Canada,Quebec,Hull,J8Y 3Z5,Male,Bachelor,Married,Star,3842.79,Standard,2014,10,False,False,103495.0


In [858]:
df_loyalty["country"].unique() #esta la borramos porque no aporta nada.

<StringArray>
['Canada']
Length: 1, dtype: str

In [859]:
df_loyalty["province"].unique()

<StringArray>
[             'Ontario',              'Alberta',     'British Columbia',
               'Quebec',                'Yukon',        'New Brunswick',
             'Manitoba',          'Nova Scotia',         'Saskatchewan',
         'Newfoundland', 'Prince Edward Island']
Length: 11, dtype: str

In [860]:
df_loyalty["city"].unique()

<StringArray>
[       'Toronto',       'Edmonton',      'Vancouver',           'Hull',
     'Whitehorse',        'Trenton',       'Montreal',   'Dawson Creek',
    'Quebec City',    'Fredericton',         'Ottawa',      'Tremblant',
        'Calgary',    'Thunder Bay',       'Whistler',    'Peace River',
       'Winnipeg',        'Sudbury', 'West Vancouver',        'Halifax',
         'London',         'Regina',        'Kelowna',     'St. John's',
       'Victoria',       'Kingston',          'Banff',        'Moncton',
  'Charlottetown']
Length: 29, dtype: str

In [861]:
df_loyalty["postal_code"].unique()

<StringArray>
['M2Z 4K1', 'T3G 6Y6', 'V6E 3D9', 'P1W 1K4', 'J8Y 3Z5', 'Y2K 6R0', 'P5S 6R4',
 'K8V 4B2', 'H2Y 2W2', 'M8Y 4K8', 'U5I 4F1', 'G1B 3L5', 'H4G 3T4', 'M2M 7K8',
 'M2M 6J7', 'E3B 2H2', 'M1R 4K3', 'T9G 1W3', 'H2Y 4R4', 'V5R 1W3', 'P1L 8X8',
 'K1F 2R2', 'H5Y 2S9', 'V1E 4R6', 'H2T 2J6', 'T3E 2V9', 'H2T 9K8', 'K8T 5M5',
 'V6T 1Y8', 'P2T 6G3', 'T9O 2W2', 'V6E 3Z3', 'R6Y 4T5', 'M5V 1G5', 'V6V 8Z3',
 'B3J 9S2', 'M5B 3E4', 'R2C 0M5', 'S6J 3G0', 'M2P 4F6', 'P1J 8T7', 'V09 2E9',
 'A1C 6H9', 'V10 6T5', 'B3C 2M8', 'M9K 2P4', 'T4V 1D4', 'R3R 3T4', 'S1J 3C5',
 'E1A 2A7', 'K1G 4Z0', 'H3T 8L4', 'C1A 6E8', 'H3J 5I6', 'M3R 4K8']
Length: 55, dtype: str

In [862]:
df_loyalty.drop(columns="country", inplace=True) #borramos columna country porque no aporta valor.


In [863]:
#Hacemos la unión usando left para asegurar que todos los registros de los vuelos se mantengan, si hay vuelos sin clientes asociados, aparecerá como valor nulo.
df_final = pd.merge(df_flight, df_loyalty,on="loyalty_number",how="left")

In [864]:
df_final.head()

,loyalty_number,year,month,flights_booked,flights_with_companions,total_flights,distance,points_accumulated,points_redeemed,dollar_cost_points_redeemed,province,city,postal_code,gender,education,marital_status,loyalty_card,clv,enrollment_type,enrollment_year,enrollment_month,cancellation_year,cancellation_month,salary
0,100018,2017,1,3,0,3,1521,152.0,0,0,Alberta,Edmonton,T9G 1W3,Female,Bachelor,Married,Aurora,7919.20,Standard,2016,8,False,False,92552.0
1,100102,2017,1,10,4,14,2030,203.0,0,0,Ontario,Toronto,M1R 4K3,Male,College,Single,Nova,2887.74,Standard,2013,3,False,False,66937.5
2,100140,2017,1,6,0,6,1200,120.0,0,0,British Columbia,Dawson Creek,U5I 4F1,Female,College,Divorced,Nova,2838.07,Standard,2016,7,False,False,66937.5
3,100214,2017,1,0,0,0,0,0.0,0,0,British Columbia,Vancouver,V5R 1W3,Male,Bachelor,Married,Star,4170.57,Standard,2015,8,False,False,63253.0
4,100272,2017,1,0,0,0,0,0.0,0,0,Ontario,Toronto,P1L 8X8,Female,Bachelor,Divorced,Star,6622.05,Standard,2014,1,False,False,91163.0


In [865]:
df_final.shape

(403760, 24)

In [866]:
df_final[df_final.duplicated()].value_counts().sum()
#no tenemos duplicados

np.int64(0)

In [867]:
df_final.isna().sum()
#no tenemos valores nulos

loyalty_number                 0
year                           0
month                          0
flights_booked                 0
flights_with_companions        0
total_flights                  0
distance                       0
points_accumulated             0
points_redeemed                0
dollar_cost_points_redeemed    0
province                       0
city                           0
postal_code                    0
gender                         0
education                      0
marital_status                 0
loyalty_card                   0
clv                            0
enrollment_type                0
enrollment_year                0
enrollment_month               0
cancellation_year              0
cancellation_month             0
salary                         0
dtype: int64

Coincide la dimensión de filas con el flight tras la eliminación de duplicados.

In [868]:
# Guardamos el resultado del merge en un archivo nuevo para seguir con la siguiente fase en un jupyter nuevo.
df_final.to_csv("files/customer_flight_loyalty.csv", index=False)